# 🧬 Lazarus — call four "dead" biology tools from your browser

> *Turn dead research code into a callable pipeline component — and give the revival back.*
>
> 🧬 **Build track** · Claude Science hackathon · July 2026 · [github.com/DoctorDean/lazarus](https://github.com/DoctorDean/lazarus)

Computational biology has a reproducibility problem: a huge share of published
methods are **open, cited, and unrunnable** within a few years. The repo is stale,
wired to a stack that no longer resolves, and the real capability is buried in
scripts with no API. The method you need *exists* — but getting it to run costs
days you don't have, so it gets abandoned.

**[Lazarus](https://github.com/DoctorDean/lazarus) is an agent that revives dead
research code, lets you compose the revivals into pipelines, and gives the fixes
back to the community.** It has autonomously resurrected four dead repos spanning
TensorFlow, CUDA, and 15-year-old C:

| Repo | Era / stack | Signature challenge it cleared alone |
|---|---|---|
| **MaSIF-site** | Py3.6 · TF 1.12 · MSMS/APBS | AVX/SIGILL/32-bit emulation walls + a silent PDB-download death |
| **ScanNet** | Py3.6 · TF 1.14 · Keras | two dead RCSB endpoints + writing its own ground-truth eval |
| **dMaSIF** | Py3.6 · torch cu111 · KeOps · **GPU** | GPU build from scratch + a CUDA source patch that *unlocked* GPU |
| **fpocket** | **2010 C** on modern GCC | download evasion + a 15-year-old undefined-behaviour bug |

### What this notebook is
A **zero-setup tour** you can run top-to-bottom right here in Colab — **no GPU, no
Docker, no accounts.** You'll run one real piece of Lazarus live (the dependency
pinner), inspect the four revived tools, and see the **rendered 3D payoff**: a
binder-triage pipeline built entirely from methods that were each unrunnable a
week ago. The last two sections show *how* the revival works and *how to run the
real compute* when you're ready.

⏱️ ~2 minutes · ▶️ *Runtime → Run all*, or step through cell by cell.


## Setup

One cell: grab Lazarus (for the code **and** the sample data) and two viewing
libraries. Nothing here needs Docker or a GPU.

In [ ]:
# Clone the repo (gives us the package + contracts + sample outputs + a structure)
import os
if not os.path.isdir("lazarus"):
    !git clone --depth 1 https://github.com/DoctorDean/lazarus.git

# Install Lazarus (base deps are tiny: packaging + PyYAML) and two viewers
!pip install -q ./lazarus py3Dmol pandas

print("\n✅ Ready.")

## 1. The core trick, *live*: commit-era dependency pinning

The single biggest reason old code "won't install" is that `pip` gives you
**today's** versions, which the old code was never written against. Lazarus
reconstructs the dependency universe **as it was on the repo's last commit**.

This is the one piece of Lazarus that runs anywhere — pure Python hitting the
PyPI API — so let's actually run it. MaSIF's README claims TensorFlow 1.9; watch
what the commit era (Jan 2019) really resolves to 👇

In [ ]:
from datetime import datetime, timezone
from lazarus.pinner import pin_requirements

as_of = datetime(2019, 1, 1, tzinfo=timezone.utc)   # MaSIF's last-commit era
pinned = pin_requirements(["tensorflow", "numpy", "scipy", "scikit-learn"], as_of)

print(f"Dependency universe as it existed on {as_of.date()}:\n")
for pkg, ver in pinned.items():
    print(f"  {pkg:14s} == {ver}")

print("\n→ tensorflow resolves to 1.12.0 — matching MaSIF's real Dockerfile,")
print("  NOT the 1.9 its README claims. That one fact unblocks the whole build.")

## 2. Four dead repos → four callable "bricks"

Every revival emits the **same integration contract**: a pinned container, a
typed input/output interface, and a **smoke test** that proves the method runs on
a fresh structure and hits a sanity threshold *you* define. That uniformity is
what makes a revived tool composable, regardless of its language or era.

Here are the four contracts Lazarus emitted — read straight from the repo:

In [ ]:
from pathlib import Path
from lazarus.contract import Contract
import pandas as pd

contracts = {
    "MaSIF-site": "lazarus/examples/masif_site_contract/lazarus.yaml",
    "ScanNet":    "lazarus/examples/scannet_ppi_contract/lazarus.yaml",
    "dMaSIF":     "lazarus/examples/dmasif_site_contract/lazarus.yaml",
    "fpocket":    "lazarus/examples/fpocket2_contract/lazarus.yaml",
}

rows = []
for label, path in contracts.items():
    c = Contract.from_yaml(Path(path).read_text())
    rows.append({
        "tool": label,
        "pinned image": c.base_image,
        "GPU": "yes" if c.gpus else "no",
        "smoke check": f"{c.smoke.metric} >= {c.smoke.threshold}" if c.smoke else "-",
        "source patches": len(c.patches),
    })

pd.set_option("display.max_colwidth", 40)
pd.DataFrame(rows)

Each contract is standalone-callable (`python predict.py structure.pdb out/`)
and works identically against a local Docker daemon or a remote one — more on that
in section 5.

## 3. The payoff: a binder-triage pipeline on PD-L1 — rendered in 3D

Because every brick speaks the same contract, you can **wire them into a pipeline**
with a few lines of YAML. This one fuses two interface predictors with a pocket
finder — three methods, three eras, one command:

```
structure ─▶ ScanNet ─┐
          ─▶ dMaSIF ──┼─▶ consensus ─▶ interface residues that also line a druggable pocket
          ─▶ fpocket ─┘
```

We run it on **PD-L1** (PDB `4ZQK`, chain A) — the immuno-oncology target. Below
is the *shipped result* of that live run (no compute needed here):

In [ ]:
print(Path("lazarus/examples/pipelines/binder_triage.yaml").read_text())

In [ ]:
print(Path("lazarus/examples/pipelines/sample_output_4ZQK/summary.txt").read_text())

### See it on the structure

Let's paint that consensus interface score straight onto PD-L1. Grey = not
predicted interface, deepening red = higher consensus interface probability.
The red residues are the predicted binding interface — exactly where the PD-1
partner docks.

In [ ]:
import py3Dmol, pandas as pd
from pathlib import Path

df = pd.read_csv("lazarus/examples/pipelines/sample_output_4ZQK/triage.csv")
pdb = Path("lazarus/pipeline-output/fpocket/4ZQK.pdb").read_text()

def score_to_hex(s):
    """Light grey (low) -> crimson (high)."""
    s = max(0.0, min(1.0, float(s)))
    lo, hi = (0xdd, 0xdd, 0xdd), (0xc0, 0x28, 0x28)
    r, g, b = (int(lo[i] + (hi[i] - lo[i]) * s) for i in range(3))
    return f"#{r:02x}{g:02x}{b:02x}"

score = dict(zip(df.resid.astype(int), df.consensus_site))

view = py3Dmol.view(width=820, height=520)
view.addModel(pdb, "pdb")
view.setStyle({"cartoon": {"color": "#dddddd"}})

# colour every scored chain-A residue by its consensus interface score
for resid, s in score.items():
    view.setStyle({"chain": "A", "resi": str(resid)},
                  {"cartoon": {"color": score_to_hex(s)}})

# draw the confident interface residues (consensus >= 0.5) as sticks
interface = df[df.consensus_site >= 0.5].resid.astype(int).tolist()
view.addStyle({"chain": "A", "resi": [str(r) for r in interface]},
              {"stick": {"colorscheme": "redCarbon", "radius": 0.18}})

view.zoomTo()
print(f"{len(interface)} predicted interface residues (consensus >= 0.5), shown as red sticks.")
view.show()

### And the pockets — why this is an *antibody* target, not a pill

`fpocket` (the 2010 C brick) did find pockets, but the deepest scores only
**0.32 druggability** — below the ~0.5 bar for a small-molecule site. Overlay the
top pocket's alpha-spheres and you can see it: the interface is broad and **flat**,
not a deep cavity a drug could wedge into.

In [ ]:
view2 = py3Dmol.view(width=820, height=520)
view2.addModel(pdb, "pdb")
view2.setStyle({"cartoon": {"color": "#dddddd"}})
view2.addStyle({"chain": "A", "resi": [str(r) for r in interface]},
               {"cartoon": {"color": "#c02828"}})

# overlay fpocket alpha-spheres for the (shallow) pockets it found
import glob
pocket_files = sorted(glob.glob("lazarus/pipeline-output/fpocket/4ZQK_out/pockets/pocket*_atm.pdb"))
colors = ["#2e86de", "#8e44ad", "#16a085"]
for i, pf in enumerate(pocket_files[:3]):
    try:
        view2.addModel(Path(pf).read_text(), "pdb")
        view2.setStyle({"model": i + 1},
                       {"sphere": {"color": colors[i % len(colors)], "opacity": 0.55, "radius": 0.9}})
    except Exception as e:
        print("skipped", pf, e)

view2.zoomTo()
print("Red = predicted interface · coloured spheres = fpocket pockets (top druggability only 0.32).")
view2.show()

> **The verdict, reproduced from dead code:** the interface is clearly
> localized but is *not* a druggable small-molecule pocket — the signature of a
> **flat protein-protein interface → an antibody / biologic target.** That's
> textbook immuno-oncology (PD-1/PD-L1 *is* an antibody target), concluded by a
> pipeline of methods that were each individually unrunnable a week ago.

## 4. How Lazarus actually revives a repo (the concept)

You never edited any of those repos by hand. Here's the loop, conceptually:

1. **Point it at a dead repo + write a goal.** The goal is a plain-English task
   *plus a sanity check* — the bar the revival must clear to count. Example (the
   real MaSIF goal file):

In [ ]:
print(Path("lazarus/examples/masif_site_goal.txt").read_text())

2. **The agent runs a repair loop in a disposable container:**
   `build → run → read the traceback → pin a dependency / patch a source bug →
   retry`, with the paper and issue tracker as context. Every expensive success
   is snapshotted, so a later failure never re-pays a slow build.
3. **It carves the capability.** It finds where *input structure → the famous
   output* actually happens and extracts the minimal path — skipping the
   hundreds-of-GB training data-prep.
4. **It emits the contract** you saw in section 2, with a smoke test that proves
   the revival on a fresh input.

```bash
# the actual command (needs Docker + Claude auth — see section 5)
lazarus resurrect --image pablogainza/masif:latest --workdir /masif \
    --goal-file examples/masif_site_goal.txt --keep
```

**These are not toy fixes.** In a single unattended run each, Lazarus:
- traced MaSIF's *silent* PDB-download death (RCSB retired the endpoint) back
  from a confusing `IndexError` pages later;
- reasoned a whole `torch 1.8.1+cu111` / PyKeOps-2.1 GPU stack for dMaSIF from a
  bare image — and **patched a CUDA tensor bug that unlocked GPU execution the
  original forced to CPU**;
- fixed a **15-year-old overlapping-`sprintf` undefined-behaviour bug** in
  fpocket's C that modern glibc silently turned into empty output.

The full gauntlet is written up in
[`docs/CHALLENGES.md`](https://github.com/DoctorDean/lazarus/blob/main/docs/CHALLENGES.md)
— roughly a person-week of specialized CUDA/glibc/TF-era debugging, cleared in
~120 autonomous agent-turns with zero human edits.

## 5. Go further: run the real compute

Everything above ran with no Docker. To actually *execute* the revived tools you
need a Docker host — and Lazarus is deliberate about **where** that host lives,
because some of these binaries (AVX-only TensorFlow, an Ampere-only GPU model)
can't run under a laptop's emulation. One flag picks the target:

| You have… | Do this | Good for |
|---|---|---|
| **Docker on your machine** (Linux/x86, or Mac for CPU-friendly bricks) | just run — local daemon is the default | fpocket, trying the flow |
| **A remote x86 / GPU box** (lab workstation, a friend's server) | `export DOCKER_HOST=ssh://you@box` — Lazarus drives it over SSH | dMaSIF's CUDA, MaSIF's MSMS |
| **A cloud VM or GPU rental** (e.g. an A-series instance) | point the *same* `DOCKER_HOST` / `--docker-host` at it | no hardware of your own |

```bash
# install with the agent extra (needs Python >= 3.10 + Docker)
pip install "lazarus-bio[agent] @ git+https://github.com/DoctorDean/lazarus"

# compose the revived bricks into the binder-triage pipeline, executed on a GPU box
lazarus run examples/pipelines/binder_triage.yaml \
    --input structure=4ZQK.pdb \
    --registry examples --registry components \
    --docker-host ssh://you@your-x86-gpu-box
```

**Auth:** Lazarus drives Claude via the
[Claude Agent SDK](https://github.com/anthropics/claude-agent-sdk-python) — log in
the `claude` CLI (subscription) or drop `ANTHROPIC_API_KEY=...` in a gitignored
`.env`. See the [README quickstart](https://github.com/DoctorDean/lazarus#quickstart).

> Colab tip: a hosted Colab runtime has no Docker daemon, so the heavy bricks
> can't run *inside* Colab — but you can point Lazarus from anywhere at a box that
> does. The notebook you're reading is the "learn it with zero setup" front door;
> `DOCKER_HOST` is the door to real compute.

## 6. …and give the revival back

A revival that dies with the hackathon isn't a fix. For the genuinely-abandoned
repos, Lazarus prepares **maintainer-ready pull requests** — the real fix plus a
CI smoke test so the method can't silently rot again:

- **MaSIF** — [PR #93](https://github.com/LPDI-EPFL/masif/pull/93): the rotted PDB download, fixed (direct RCSB fetch) + CI.
- **ScanNet** — [PR #16](https://github.com/jertubiana/ScanNet/pull/16): `library_folder=''` made to auto-detect the repo root + CI.

*(dMaSIF is skipped — no-derivatives license; fpocket's upstream is alive.)*

### Explore more
- 📦 **Repo:** [github.com/DoctorDean/lazarus](https://github.com/DoctorDean/lazarus)
- 🧗 **The hard problems it solved:** [`docs/CHALLENGES.md`](https://github.com/DoctorDean/lazarus/blob/main/docs/CHALLENGES.md)
- 📊 **Three-way method comparison:** [`analysis/RESULTS.md`](https://github.com/DoctorDean/lazarus/blob/main/analysis/RESULTS.md)
- ✅ **Reproduces the paper:** MaSIF-site transient benchmark, 0.82 vs paper's 0.85

**Lazarus takes a method you couldn't run this morning and hands you back a brick
you can call this afternoon — then mails the fix upstream.**
